In [10]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

import matplotlib.pyplot as plt
from binance.client import Client
from datetime import datetime, timezone
import os

In [11]:


SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_FRAC = 0.8

# Trading / reward parameters
FEE = 0.0005         # transaction cost per unit position change
KAPPA = 0.1          # risk penalty weight
INITIAL_BUDGET = 100000.0  # Starting capital

# PPO hyperparameters - OPTIMIZED TO PREVENT OVERFITTING
num_envs = 4  # Reduced from 16 for stability on CPU
n_steps = 128  # Good rollout length
total_updates = 700  # EARLY STOP before overfitting (peaks at ~700, then degrades)

gamma = 0.99
gae_lambda = 0.95

lr = 1e-4  # Slower learning (was 3e-4) - prevents overshooting and instability
vf_coef = 0.5
ent_coef = 0.01  # 10x higher (was 0.001) - MORE EXPLORATION to prevent overfitting
max_grad_norm = 0.5

clip_eps = 0.1  # More conservative (was 0.2) - prevents aggressive policy updates
ppo_epochs = 4  # Fewer epochs (was 10) - less overtraining per update
minibatch_size = 32  # Smaller batches (was 64) - more stable gradient
target_kl = 0.1

# ============================================================================
# DATA LOADING FROM BINANCE WITH CACHE
# ============================================================================
print("Loading BTCUSDT data...")
cache_file = "btcusdt_cache.csv"

if os.path.exists(cache_file):
    print("  Loading from cache...")
    df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
    print(f"  ✅ Loaded {len(df)} rows")
else:
    print("  Fetching from Binance API...")
    client = Client()
    klines = client.get_historical_klines(
        "BTCUSDT", Client.KLINE_INTERVAL_1DAY, "1 Jan, 2023",
        datetime.now(timezone.utc).strftime("%d %b, %Y")
    )
    df = pd.DataFrame(klines, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base", "taker_buy_quote", "ignore"
    ])
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df = df.set_index("open_time")
    df = df[["open", "high", "low", "close", "volume"]].astype(float).dropna()
    df.to_csv(cache_file)
    print(f"  ✅ Loaded {len(df)} rows and cached")

print()

Loading BTCUSDT data...
  Loading from cache...
  ✅ Loaded 1159 rows



# Feature engineering + Bitcoin-optimized technical indicators

In [12]:
print("Computing features...")

def add_features_and_forecast(df, ewma_span=20, vol_window=20):
    df = df.copy()
    df["log_close"] = np.log(df["close"])
    df["r"] = df["log_close"].diff()
    df["mu_hat"] = df["r"].ewm(span=ewma_span, adjust=False).mean()
    df["sigma_hat"] = df["r"].rolling(vol_window).std()
    df["r_lag1"] = df["r"].shift(1)

    # RSI (14-period)
    def compute_rsi(series, period=14):
        delta = series.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-8)
        return (100 - (100 / (1 + rs))) / 100.0

    df["rsi"] = compute_rsi(df["close"], period=14)

    # MACD
    ema_12 = df["close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["close"].ewm(span=26, adjust=False).mean()
    df["macd"] = ema_12 - ema_26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_norm"] = df["macd"] / (df["close"] + 1e-8)

    # Bollinger Bands
    sma_20 = df["close"].rolling(window=20).mean()
    std_20 = df["close"].rolling(window=20).std()
    df["bb_position"] = (df["close"] - (sma_20 - 2*std_20)) / (4*std_20 + 1e-8)
    df["bb_position"] = df["bb_position"].clip(0, 1)

    # SMA 20 & 200
    df["sma_20"] = sma_20
    df["sma_200"] = df["close"].rolling(window=200).mean()
    df["sma_ratio"] = (df["sma_20"] / (df["sma_200"] + 1e-8) - 1.0).clip(-0.5, 0.5)
    df["price_sma20_dist"] = ((df["close"] - sma_20) / (sma_20 + 1e-8)).clip(-0.1, 0.1)

    # ATR
    high_low = df["high"] - df["low"]
    high_close = abs(df["high"] - df["close"].shift())
    low_close = abs(df["low"] - df["close"].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["atr_ratio"] = (true_range.rolling(14).mean() / (df["close"] + 1e-8)).clip(0, 0.05)

    # Volume
    df["volume_ma_20"] = df["volume"].rolling(window=20).mean()
    df["volume_ratio"] = np.log1p((df["volume"] / (df["volume_ma_20"] + 1e-8)).clip(0, 5))

    # Golden Cross
    df["golden_cross"] = (df["sma_20"] > df["sma_200"]).astype(float)

    # Momentum
    df["momentum_5"] = df["close"].pct_change(5).clip(-0.1, 0.1)

    return df.dropna()

df_feat = add_features_and_forecast(df)
print(f"✅ {len(df_feat.columns)} features created\n")

# CRITICAL: Reset index to ensure integer-based indexing works!
df_feat = df_feat.reset_index(drop=True)
print(f"✅ Index reset - shape: {df_feat.shape}\n")
df_feat.head()

Computing features...
✅ 24 features created

✅ Index reset - shape: (960, 24)



,open,high,low,close,volume,log_close,r,mu_hat,sigma_hat,r_lag1,...,bb_position,sma_20,sma_200,sma_ratio,price_sma20_dist,atr_ratio,volume_ma_20,volume_ratio,golden_cross,momentum_5
0,29859.14,30189.09,29761.96,29909.21,25657.36137,10.305922,0.001676,-0.000684,0.014590,-0.009296,...,0.175282,30419.7840,25923.93860,0.173424,-0.016784,0.026726,36502.865424,0.532325,1.0,-0.013288
1,29909.21,30417.46,29570.96,29800.00,37540.68193,10.302264,-0.003658,-0.000967,0.014596,0.001676,...,0.148082,30386.1840,25989.85485,0.169156,-0.019291,0.024899,33908.945711,0.745314,1.0,-0.016161
2,29800.00,30061.70,29726.34,29901.72,23881.40865,10.305671,0.003408,-0.000551,0.014591,-0.003658,...,0.236456,30351.9750,26055.99910,0.164875,-0.014834,0.023829,34227.928606,0.529284,1.0,-0.010925
3,29901.72,29999.00,29625.10,29794.00,14660.40467,10.302062,-0.003609,-0.000842,0.014592,0.003408,...,0.206148,30310.8235,26121.59320,0.160374,-0.017051,0.023990,33796.628330,0.360316,1.0,-0.011414
4,29793.99,30350.00,29730.00,30083.75,18292.78637,10.311740,0.009678,0.000160,0.014161,-0.003609,...,0.390003,30257.2010,26187.76015,0.155395,-0.005733,0.024319,32523.185494,0.446257,1.0,0.007523


# Train/Test split (time-based)

In [13]:
n = len(df_feat)
split = int(TRAIN_FRAC * n)

df_train = df_feat.iloc[:split].reset_index(drop=True)
df_test  = df_feat.iloc[split:].reset_index(drop=True)

print(len(df_train), len(df_test))

768 192


# Trading Environment (target position action)

In [14]:
class TradingEnv(gym.Env):
    """
    Enhanced trading environment with budget/liquidity tracking and refined long/short.
    - Action: target position a_t in [-1, 1]
    - Reward: pnl - transaction_cost - risk_penalty
    - State: market features + portfolio features (including budget/liquidity)
    - Enhanced with cumulative profit tracking and refined leverage constraints
    """
    metadata = {"render_modes": []}

    def __init__(self, df, fee=0.0005, kappa=0.1, initial_budget=100000.0, max_leverage=2.0):
        super().__init__()
        self.df = df.reset_index(drop=True)  # CRITICAL: reset index!
        self.fee = float(fee)
        self.kappa = float(kappa)
        self.initial_budget = float(initial_budget)
        self.max_leverage = float(max_leverage)

        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)

        # Use ONLY features that definitely exist
        base_features = ["r", "r_lag1", "mu_hat", "sigma_hat"]
        optional_features = ["rsi", "macd_norm", "bb_position", "sma_ratio",
                            "price_sma20_dist", "atr_ratio", "volume_ratio",
                            "golden_cross", "momentum_5"]

        # Build feature list - only include features that exist
        self.feature_cols = [f for f in base_features if f in df.columns]
        self.feature_cols += [f for f in optional_features if f in df.columns]

        print(f"Using {len(self.feature_cols)} features: {self.feature_cols}")

        # obs_dim = market_features + portfolio_features(11)
        obs_dim = len(self.feature_cols) + 11
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 1  # start from 1 because we use r_t
        self.pos = 0.0  # Position (fraction of capital)
        self.equity = self.initial_budget
        self.initial_equity = self.initial_budget
        self.peak = self.initial_budget
        self.cumulative_profit = 0.0  # Track absolute profit
        self.cost_accumulated = 0.0   # Track total transaction costs

        # Additional tracking for enhanced state
        self.trades_made = 0
        self.winning_trades = 0
        self.cumulative_returns = 0.0
        self.position_entry_time = None

        return self._get_obs(), {}

    def _get_obs(self):
        """Safe observation - robust bounds checking and error handling"""

        # CRITICAL BOUNDS CHECK
        if len(self.df) == 0:
            raise RuntimeError("DataFrame is empty!")

        # Clamp time index to valid range
        t_idx = int(np.clip(self.t, 0, len(self.df) - 1))

        # Get market features SAFELY
        market_features = []
        for col in self.feature_cols:
            try:
                val = float(self.df.iloc[t_idx][col])
                # Handle NaN/Inf
                if np.isnan(val) or np.isinf(val):
                    val = 0.0
            except (KeyError, IndexError, ValueError, TypeError):
                # If feature doesn't exist or has error, use 0.0
                val = 0.0
            market_features.append(val)

        x = np.array(market_features, dtype=np.float32)

        # Portfolio features
        equity_norm = float(self.equity / self.initial_budget)
        drawdown = float((self.peak - self.equity) / max(self.peak, 1.0 + 1e-8))
        position_value = abs(self.pos) * self.equity
        free_capital = max(0.0, self.equity - position_value)
        liquidity_ratio = float(free_capital / max(self.equity, 1.0 + 1e-8))
        leverage = float(abs(self.pos) * self.max_leverage)

        # Unrealized PnL - SAFE
        try:
            r_t_current = float(self.df.iloc[t_idx]["r"])
            if not np.isfinite(r_t_current):
                r_t_current = 0.0
        except:
            r_t_current = 0.0
        unrealized_pnl = float(self.pos * r_t_current)

        cumulative_returns = float(self.cumulative_returns)
        momentum = float(0.0)
        win_rate = float(self.winning_trades / max(self.trades_made, 1.0) if self.trades_made > 0 else 0.0)
        time_in_position = float(0.0)
        costs_norm = float(min(1.0, self.cost_accumulated / max(self.initial_budget, 1.0)))

        portfolio_features = np.array([
            self.pos, equity_norm, drawdown, liquidity_ratio, leverage,
            unrealized_pnl, cumulative_returns, momentum, win_rate,
            time_in_position, costs_norm
        ], dtype=np.float32)

        obs = np.concatenate([x, portfolio_features])
        return obs

    def step(self, action):
        # Safety check: if we're at the end, return dummy obs and terminate
        if self.t >= len(self.df) - 1:
            return self._get_obs(), 0.0, True, False, {
                "cumulative_profit": self.cumulative_profit,
                "equity": self.equity,
                "position": self.pos,
                "costs": self.cost_accumulated
            }

        # Get data BEFORE incrementing time - use iloc for safe indexing
        r_t = float(self.df.iloc[self.t]["r"])
        sigma_t = float(self.df.iloc[self.t]["sigma_hat"])
        if not np.isfinite(sigma_t):
            sigma_t = 0.01

        # Clip action to valid range
        a = float(np.clip(action[0], -1.0, 1.0))

        # === REWARD ===
        pnl_reward = 100.0 * self.pos * r_t
        position_change = abs(a - self.pos)
        cost_penalty = 0.1 * self.fee * position_change
        risk_penalty = 0.01 * (a ** 2) * sigma_t
        reward = pnl_reward - cost_penalty - risk_penalty

        # === UPDATE EQUITY ===
        pnl_dollars = self.pos * self.equity * r_t
        transaction_cost_dollars = cost_penalty * self.equity
        self.equity = self.equity + pnl_dollars - transaction_cost_dollars

        if self.equity < 0:
            self.equity = self.initial_budget * 0.001
            reward = -1.0

        # === TRACKING ===
        self.cumulative_profit += pnl_dollars - transaction_cost_dollars
        self.cost_accumulated += transaction_cost_dollars
        self.cumulative_returns += pnl_reward

        if position_change > 0.01:
            self.trades_made += 1
            if pnl_dollars > 0:
                self.winning_trades += 1
            self.position_entry_time = self.t

        # === UPDATE STATE ===
        self.pos = a
        self.peak = max(self.peak, self.equity)

        # === INCREMENT TIME (AFTER getting data) ===
        self.t += 1

        # === CHECK TERMINATION ===
        terminated = (self.t >= len(self.df) - 1) or (self.equity <= 0)
        truncated = False

        # === GET NEXT OBS (with safe bounds checking) ===
        # If we're at the end, return dummy observation
        if self.t >= len(self.df):
            next_obs = self._get_obs()  # Will clamp to last valid index
        else:
            next_obs = self._get_obs()

        return next_obs, float(reward), terminated, truncated, {
            "cumulative_profit": self.cumulative_profit,
            "equity": self.equity,
            "position": self.pos,
            "costs": self.cost_accumulated
        }

# Vectorized env (train)

In [15]:
def make_env(df):
    def thunk():
        return TradingEnv(df, fee=FEE, kappa=KAPPA, initial_budget=INITIAL_BUDGET, max_leverage=2.0)
    return thunk

env = gym.vector.SyncVectorEnv([make_env(df_train) for _ in range(num_envs)])
obs_dim = env.single_observation_space.shape[0]
act_dim = env.single_action_space.shape[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("obs_dim:", obs_dim, "act_dim:", act_dim, "device:", device)

Using 13 features: ['r', 'r_lag1', 'mu_hat', 'sigma_hat', 'rsi', 'macd_norm', 'bb_position', 'sma_ratio', 'price_sma20_dist', 'atr_ratio', 'volume_ratio', 'golden_cross', 'momentum_5']
Using 13 features: ['r', 'r_lag1', 'mu_hat', 'sigma_hat', 'rsi', 'macd_norm', 'bb_position', 'sma_ratio', 'price_sma20_dist', 'atr_ratio', 'volume_ratio', 'golden_cross', 'momentum_5']
Using 13 features: ['r', 'r_lag1', 'mu_hat', 'sigma_hat', 'rsi', 'macd_norm', 'bb_position', 'sma_ratio', 'price_sma20_dist', 'atr_ratio', 'volume_ratio', 'golden_cross', 'momentum_5']
Using 13 features: ['r', 'r_lag1', 'mu_hat', 'sigma_hat', 'rsi', 'macd_norm', 'bb_position', 'sma_ratio', 'price_sma20_dist', 'atr_ratio', 'volume_ratio', 'golden_cross', 'momentum_5']
obs_dim: 24 act_dim: 1 device: cpu


# PPO model (Gaussian policy + tanh squash + corrected logprob)

In [16]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh()
        )
        self.mu = nn.Linear(128, act_dim)
        self.log_std = nn.Parameter(torch.ones(act_dim) * -1.0)  # good start
        self.v = nn.Linear(128, 1)

    def forward(self, obs):
        x = self.net(obs)
        mu = self.mu(x)
        std = torch.exp(self.log_std)
        dist = Normal(mu, std)
        value = self.v(x).squeeze(-1)
        return dist, value

def squash(u):
    return torch.tanh(u)  # maps to [-1,1]
# main formula:
# a = f(u)
# log p(a) = log p(u) - log |det(Jacobian)|
# log p(a) ist die gesuchte policy log pi(a|a)

# we have f = tanh
# a = tanh(u)
# da/du = 1 - tanh(u)^2
# da/du = 1 - a²
# we need: log |det(Jacobian)|
# we get: log |det(Jacobian)| = log(1 - tanh(u)^2)
# in code: log_det = torch.log(1.0 - torch.tanh(u).pow(2) + eps).sum(-1)

def logprob_squashed(dist, u):
    # log p(u)
    logp_u = dist.log_prob(u).sum(-1)
    # change-of-variables for tanh
    eps = 1e-6
    log_det = torch.log(1.0 - torch.tanh(u).pow(2) + eps).sum(-1)
    return logp_u - log_det

# GAE

In [17]:
def compute_gae(rewards, dones, values, last_value, gamma=0.99, lam=0.95):
    """
    rewards: [T, N]
    dones:   [T, N] (1.0 means terminal boundary for bootstrap mask)
    values:  [T, N]
    last_value: [N]
    """
    T, N = rewards.shape
    adv = torch.zeros(T, N, device=values.device)
    gae = torch.zeros(N, device=values.device)

    for t in reversed(range(T)):
        not_done = 1.0 - dones[t]
        next_value = last_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_value * not_done - values[t]
        gae = delta + gamma * lam * not_done * gae
        adv[t] = gae

    returns = adv + values
    return returns, adv

# PPO training loop

In [18]:
model = ActorCritic(obs_dim, act_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

obs, _ = env.reset(seed=SEED)
# Keep obs as numpy array - convert to tensor when needed in the loop
obs = obs  # obs is numpy from env.reset()

ep_returns = np.zeros(num_envs, dtype=np.float32)
ep_history = []

for update in range(total_updates):
    # Rollout buffers
    obs_buf  = torch.zeros(n_steps, num_envs, obs_dim, device=device)
    u_buf    = torch.zeros(n_steps, num_envs, act_dim, device=device)
    logp_buf = torch.zeros(n_steps, num_envs, device=device)
    rew_buf  = torch.zeros(n_steps, num_envs, device=device)
    done_buf = torch.zeros(n_steps, num_envs, device=device)
    val_buf  = torch.zeros(n_steps, num_envs, device=device)

    for t in range(n_steps):
        # Convert numpy obs to torch if needed
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device) if isinstance(obs, np.ndarray) else obs
        obs_buf[t] = obs_tensor

        with torch.no_grad():
          dist, value = model(obs_tensor)
          u = dist.sample()
          a = squash(u)
          logp = logprob_squashed(dist, u)

        u_buf[t] = u
        logp_buf[t] = logp.detach()
        val_buf[t] = value.detach()

        # Action shape is (num_envs, act_dim) - make sure to flatten correctly
        actions = a.detach().cpu().numpy()  # Shape: (num_envs, act_dim)

        next_obs, reward, terminated, truncated, infos = env.step(actions)
        done_env = np.logical_or(terminated, truncated)
        done_boot = terminated  # bootstrap mask (for time-limit envs you may choose terminated only)

        rew_buf[t] = torch.as_tensor(reward, dtype=torch.float32, device=device)
        done_buf[t] = torch.as_tensor(done_boot, dtype=torch.float32, device=device)

        # Episode return tracking
        ep_returns += reward
        if done_env.any():
            finished = np.where(done_env)[0]
            ep_history.extend(ep_returns[finished].tolist())
            ep_returns[finished] = 0.0

        obs = next_obs  # Keep as numpy for next iteration

    # Bootstrap last value
    with torch.no_grad():
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device) if isinstance(obs, np.ndarray) else obs
        _, last_value = model(obs_tensor)

    returns, adv = compute_gae(rew_buf, done_buf, val_buf, last_value, gamma=gamma, lam=gae_lambda)

    # Flatten
    B = n_steps * num_envs
    obs_batch  = obs_buf.reshape(B, obs_dim)
    u_batch    = u_buf.reshape(B, act_dim)
    old_logp   = logp_buf.reshape(B)
    old_value  = val_buf.reshape(B)      # important for value clipping if you add it later
    ret_batch  = returns.reshape(B).detach()
    adv_batch  = adv.reshape(B).detach()

    # Advantage normalization
    adv_batch = (adv_batch - adv_batch.mean()) / (adv_batch.std() + 1e-8)

    idx = torch.arange(B, device=device)
    stop = False

    for _ in range(ppo_epochs):
        perm = idx[torch.randperm(B)]
        for start in range(0, B, minibatch_size):
            mb = perm[start:start + minibatch_size]

            dist, value = model(obs_batch[mb])
            logp = logprob_squashed(dist, u_batch[mb])
            entropy = dist.entropy().sum(-1)

            # Early stop by approximate KL (minibatch estimate)
            approx_kl = (old_logp[mb] - logp).mean().detach()
            if approx_kl.item() > target_kl:
                stop = True
                break

            ratio = torch.exp(logp - old_logp[mb])

            # Clipped policy objective
            unclipped = ratio * adv_batch[mb]
            clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv_batch[mb]
            policy_loss = -torch.min(unclipped, clipped).mean()

            # Value loss (simple version; you can add value clipping later)
            value_loss = (ret_batch[mb] - value).pow(2).mean()

            # Entropy bonus
            entropy_loss = -entropy.mean()

            loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        if stop:
            break

        # Keep std in a sane range (helps prevent wild exploration collapse/explosion)
        with torch.no_grad():
            model.log_std.clamp_(-2.0, -0.5)

    if update % 100 == 0:
        mean_100 = np.mean(ep_history[-100:]) if len(ep_history) >= 100 else np.nan
        print(f"Update {update:4d} | mean_return(last100) {mean_100:8.2f} | log_std {model.log_std.data.cpu().numpy()} | eps {clip_eps:.2f} | lr {lr:.2e}")

        # DEBUG: Show first episode's reward structure (only on first update)
        if update == 0 and len(ep_history) < 10:
            print("\n[DEBUG] First episode analysis:")
            try:
                test_env = TradingEnv(df_train[:100], fee=FEE, kappa=KAPPA, initial_budget=INITIAL_BUDGET)
                obs, _ = test_env.reset()
                for i in range(10):
                    a = np.array([0.5])  # Test: long position
                    obs, rew, term, trunc, info = test_env.step(a)
                    # Use safe index access
                    r_t = float(test_env.df.iloc[max(0, test_env.t-2)]["r"]) if test_env.t >= 2 else 0.0
                    print(f"  Step {i}: pos={test_env.pos:.2f}, r_t={r_t:.4f}, reward={rew:.6f}, equity=${test_env.equity:,.0f}")
                    if term or trunc:
                        break
                print()
            except Exception as e:
                print(f"  [DEBUG] Error: {e}\n")

Update    0 | mean_return(last100)      nan | log_std [-0.99976504] | eps 0.10 | lr 1.00e-04

[DEBUG] First episode analysis:
Using 13 features: ['r', 'r_lag1', 'mu_hat', 'sigma_hat', 'rsi', 'macd_norm', 'bb_position', 'sma_ratio', 'price_sma20_dist', 'atr_ratio', 'volume_ratio', 'golden_cross', 'momentum_5']
  Step 0: pos=0.50, r_t=0.0017, reward=-0.000061, equity=$99,998
  Step 1: pos=0.50, r_t=-0.0037, reward=0.170344, equity=$100,168
  Step 2: pos=0.50, r_t=0.0034, reward=-0.180485, equity=$99,987
  Step 3: pos=0.50, r_t=-0.0036, reward=0.483871, equity=$100,471
  Step 4: pos=0.50, r_t=0.0097, reward=-1.531117, equity=$98,933
  Step 5: pos=0.50, r_t=-0.0306, reward=0.089696, equity=$99,021
  Step 6: pos=0.50, r_t=0.0018, reward=0.210015, equity=$99,229
  Step 7: pos=0.50, r_t=0.0042, reward=-0.220575, equity=$99,011
  Step 8: pos=0.50, r_t=-0.0044, reward=0.156037, equity=$99,165
  Step 9: pos=0.50, r_t=0.0031, reward=0.066032, equity=$99,231



IndexError: invalid index to scalar variable.

# Evaluation on test set (single env, deterministic actions)

In [ ]:
def eval_policy(model, df_eval, episodes=5):
    env_eval = TradingEnv(df_eval, fee=FEE, kappa=KAPPA, initial_budget=INITIAL_BUDGET, max_leverage=2.0)
    returns = []

    for _ in range(episodes):
        obs, _ = env_eval.reset()
        done = False
        ep_ret = 0.0

        while not done:
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                dist, _ = model(obs_t)
                # Deterministic: use mean action (mu), then squash
                u = dist.mean
                a = squash(u).cpu().numpy()[0]

            obs, reward, terminated, truncated, _ = env_eval.step(a)
            done = terminated or truncated
            ep_ret += reward

        returns.append(ep_ret)

    return float(np.mean(returns))

test_score = eval_policy(model, df_test, episodes=10)
print("EVAL mean episode reward:", test_score)

# Equity curve plot (test set, one run)

In [ ]:
def run_equity_curve(model, df_eval):
    env_eval = TradingEnv(df_eval, fee=FEE, kappa=KAPPA, initial_budget=INITIAL_BUDGET, max_leverage=2.0)
    obs, _ = env_eval.reset()
    done = False

    equity = [env_eval.equity]
    pos_hist = [env_eval.pos]
    profit_hist = [0.0]  # Track cumulative profit
    cost_hist = [0.0]    # Track cumulative costs

    while not done:
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            dist, _ = model(obs_t)
            u = dist.mean
            a = squash(u).cpu().numpy()[0]

        obs, reward, terminated, truncated, info = env_eval.step(a)
        done = terminated or truncated
        equity.append(env_eval.equity)
        pos_hist.append(env_eval.pos)
        profit_hist.append(info["cumulative_profit"])
        cost_hist.append(info["costs"])

    return np.array(equity), np.array(pos_hist), np.array(profit_hist), np.array(cost_hist)

equity, pos_hist, profit_hist, cost_hist = run_equity_curve(model, df_test)

# Plotting
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Equity Curve
axes[0, 0].plot(equity, linewidth=2, color='green')
axes[0, 0].fill_between(range(len(equity)), INITIAL_BUDGET, equity, alpha=0.3, color='green')
axes[0, 0].axhline(y=INITIAL_BUDGET, color='red', linestyle='--', label='Initial Budget')
axes[0, 0].set_title("Equity Curve (Test Set)", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Trading Days")
axes[0, 0].set_ylabel("Equity ($)")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Cumulative Profit
axes[0, 1].plot(profit_hist, linewidth=2, color='blue')
axes[0, 1].fill_between(range(len(profit_hist)), 0, profit_hist, alpha=0.3, color='blue')
axes[0, 1].set_title("Cumulative Profit (Test Set)", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Trading Days")
axes[0, 1].set_ylabel("Profit ($)")
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Position History
axes[1, 0].plot(pos_hist, linewidth=1.5, color='purple')
axes[1, 0].fill_between(range(len(pos_hist)), 0, pos_hist, alpha=0.3, color='purple')
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 0].set_title("Position History (Test Set)", fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel("Trading Days")
axes[1, 0].set_ylabel("Position [-1 (Short), +1 (Long)]")
axes[1, 0].set_ylim([-1.2, 1.2])
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Cumulative Costs
axes[1, 1].plot(cost_hist, linewidth=2, color='red')
axes[1, 1].fill_between(range(len(cost_hist)), 0, cost_hist, alpha=0.3, color='red')
axes[1, 1].set_title("Cumulative Transaction Costs (Test Set)", fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel("Trading Days")
axes[1, 1].set_ylabel("Costs ($)")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print Summary Statistics
print("=" * 70)
print("TRADING PERFORMANCE SUMMARY")
print("=" * 70)
print(f"Initial Budget:         ${INITIAL_BUDGET:>15,.2f}")
print(f"Final Equity:           ${equity[-1]:>15,.2f}")
print(f"Total Profit:           ${profit_hist[-1]:>15,.2f}")
print(f"Total Costs:            ${cost_hist[-1]:>15,.2f}")
print(f"Return:                 {(equity[-1] - INITIAL_BUDGET) / INITIAL_BUDGET * 100:>15.2f}%")
print(f"Max Drawdown:           {(1 - (equity.min() / INITIAL_BUDGET)) * 100:>15.2f}%")
print(f"Sharpe Ratio:           {np.std(np.diff(equity)) / (np.mean(np.diff(equity)) + 1e-8) if np.mean(np.diff(equity)) != 0 else 0:>15.4f}")

# Additional metrics
win_days = np.sum(np.diff(equity) > 0)
total_days = len(np.diff(equity))
win_rate = win_days / total_days * 100 if total_days > 0 else 0
avg_daily_return = np.mean(np.diff(equity)) if len(np.diff(equity)) > 0 else 0
daily_volatility = np.std(np.diff(equity)) if len(np.diff(equity)) > 0 else 0

print(f"Win Rate:               {win_rate:>15.2f}%")
print(f"Avg Daily PnL:          ${avg_daily_return:>15,.2f}")
print(f"Daily Volatility:       ${daily_volatility:>15,.2f}")
print(f"Cost/Profit Ratio:      {cost_hist[-1] / (abs(profit_hist[-1]) + 1e-8):>15.4f}")

# Leverage analysis
avg_leverage = np.mean(np.abs(pos_hist))
max_leverage = np.max(np.abs(pos_hist))
print(f"Avg Leverage:           {avg_leverage:>15.2f}x")
print(f"Max Leverage:           {max_leverage:>15.2f}x")

print("=" * 70)
